# SigAlg's `Filtration` class

In [1]:
# If running in Google Colab, uncomment the line below and run this cell first
# !pip install sigalg

The `Filtration` class in SigAlg represents a filtration of $\sigma$-algebras on a sample space. The API reference is [here](https://johnmyers-phd.com/sigalg/api/core/#sigalg.core.Filtration){target="_blank"}.

## Mathematical definition

A $\sigma$-algebra $\mathcal{F}$ on a set $\Omega$ is called a *filtered $\sigma$-algebra* if it equipped with a collection $\{\mathcal{F}_t\}_{t\in T}$ of $\sigma$-algebras on $\Omega$, indexed by some linearly ordered set $T$, such that $\mathcal{F}_t \subset \mathcal{F}$ for every $t\in T$, and $\mathcal{F}_s \subset \mathcal{F}_t$ for all $s,t\in T$ with $s\leq t$. In this case, the collection $\{\mathcal{F}_t\}_{t\in T}$ is called a *filtration*.

In SigAlg, an instance `F` of `Filtration` represents such a filtration. The instance carries:
- A `time` attribute representing the index set $T$
- A `sigma_algebras` attribute (a list) containing the $\sigma$-algebras $\mathcal{F}_t$
- A `data` attribute (a `pd.DataFrame`) where each column represents a $\sigma$-algebra in the filtration

## API examples

### Creating filtrations

#### From a list of $\sigma$-algebras

Begin by defining a sample space $\Omega = \{0,1,2,3\}$ and creating a sequence of nested $\sigma$-algebras:

In [2]:
from sigalg.core import Filtration, SampleSpace, SigmaAlgebra, Time

Omega = SampleSpace().from_sequence(size=4)

print(Omega)

Sample space 'Omega':
[0, 1, 2, 3]


Create three $\sigma$-algebras that form a nested sequence (trivial -> intermediate -> power set):

In [3]:
# Coarsest: trivial sigma-algebra
F_0 = SigmaAlgebra.trivial(sample_space=Omega, name="F_0")

# Intermediate: partition into {0,1} and {2,3}
F_1 = SigmaAlgebra(sample_space=Omega, name="F_1").from_dict(
    {
        0: 0,
        1: 0,
        2: 1,
        3: 1,
    }
)

# Finest: power-set sigma-algebra
F_2 = SigmaAlgebra.power_set(sample_space=Omega, name="F_2")

print("F_0 (trivial):")
print(F_0, "\n")
print("F_1 (intermediate):")
print(F_1, "\n")
print("F_2 (power set):")
print(F_2)

F_0 (trivial):
Sigma algebra 'F_0':
        atom ID
sample         
0             0
1             0
2             0
3             0 

F_1 (intermediate):
Sigma algebra 'F_1':
        atom ID
sample         
0             0
1             0
2             1
3             1 

F_2 (power set):
Sigma algebra 'F_2':
        atom ID
sample         
0             0
1             1
2             2
3             3


Create a filtration from this list:

In [4]:
time = Time.discrete(length=2)
F = Filtration(time=time, name="F").from_list([F_0, F_1, F_2])

print(F)

Filtration 'F'

* Time 'T':
[0, 1, 2]

* At index 0:
Sigma algebra 'F_0':
        atom ID
sample         
0             0
1             0
2             0
3             0

* At index 1:
Sigma algebra 'F_1':
        atom ID
sample         
0             0
1             0
2             1
3             1

* At index 2:
Sigma algebra 'F_2':
        atom ID
sample         
0             0
1             1
2             2
3             3


If time is not provided, it will be automatically generated:

In [5]:
G = Filtration(name="G").from_list([F_0, F_1, F_2])
print(f"Time index:\n{G.time}")

Time index:
Time 'T':
[0, 1, 2, 3]


#### From a `pd.DataFrame`

Create a filtration from a data frame where each column represents a $\sigma$-algebra:

In [6]:
import pandas as pd

df = pd.DataFrame(
    [
        [0, 0, 0, 0],
        [0, 0, 1, 1],
        [0, 1, 2, 3],
    ],
    index=["a", "b", "c"],
)

H = Filtration(name="H").from_pandas(df)
print(H)

Filtration 'H'

* Time 'T':
[0, 1, 2, 3]

* At index 0:
Sigma algebra '0':
        atom ID
sample         
a             0
b             0
c             0

* At index 1:
Sigma algebra '1':
        atom ID
sample         
a             0
b             0
c             1

* At index 2:
Sigma algebra '2':
        atom ID
sample         
a             0
b             1
c             2

* At index 3:
Sigma algebra '3':
        atom ID
sample         
a             0
b             1
c             3


### Properties of filtrations

#### Time index

Access the time index of the filtration:

In [7]:
time = Time.continuous(start=0, stop=2, num_points=3)
F = Filtration(time=time, name="F").from_list([F_0, F_1, F_2])

print(f"Time index:\n{F.time}\n")
print(f"Time to position mapping:\n{F.time_to_pos}")

Time index:
Time 'T':
[0.0, 1.0, 2.0]

Time to position mapping:
{0.0: 0, 1.0: 1, 2.0: 2}


#### Data

Access the underlying data frame:

In [8]:
print(F.data)

        F_0  F_1  F_2
sample               
0         0    0    0
1         0    0    1
2         0    1    2
3         0    1    3


#### Coarsest and finest $\sigma$-algebras

Access the coarsest and finest sigma-algebras in the filtration:

In [9]:
print("Coarsest sigma-algebra:")
print(F.coarsest, "\n")
print("Finest sigma-algebra:")
print(F.finest)

Coarsest sigma-algebra:
Sigma algebra 'F_0':
        atom ID
sample         
0             0
1             0
2             0
3             0 

Finest sigma-algebra:
Sigma algebra 'F_2':
        atom ID
sample         
0             0
1             1
2             2
3             3


#### Sample space

All $\sigma$-algebras in a filtration share the same sample space:

In [10]:
print(F.sample_space)

Sample space 'Omega':
[0, 1, 2, 3]


#### Length

Get the number of $\sigma$-algebras in the filtration:

In [11]:
print(f"Length of filtration: {len(F)}")

Length of filtration: 3


### Accessing $\sigma$-algebras

#### Indexing by time

Access the $\sigma$-algebra at a specific time point:

In [12]:
print("Sigma-algebra at time 0.0:")
print(F[0.0], "\n")
print("Sigma-algebra at time 1.0:")
print(F[1.0])

Sigma-algebra at time 0.0:
Sigma algebra 'F_0':
        atom ID
sample         
0             0
1             0
2             0
3             0 

Sigma-algebra at time 1.0:
Sigma algebra 'F_1':
        atom ID
sample         
0             0
1             0
2             1
3             1


If the filtration is indexed by continuous time (as the current example `F` is), then indexing into the filtration using a time value that is *not* explicitly in the index returns the $\sigma$-algebra at the nearest time:

In [13]:
print(f"The actual time index:\n{F.time}\n")

print(f"Indexing at a time t=0.25 not explicitly in the index returns F_0:\n{F[0.25]}")

The actual time index:
Time 'T':
[0.0, 1.0, 2.0]

Indexing at a time t=0.25 not explicitly in the index returns F_0:
Sigma algebra 'F_0':
        atom ID
sample         
0             0
1             0
2             0
3             0


#### Iterating over $\sigma$-algebras

Iterate through all $\sigma$-algebras in the filtration:

In [14]:
for sig_alg in F:
    print(sig_alg, "\n")

Sigma algebra 'F_0':
        atom ID
sample         
0             0
1             0
2             0
3             0 

Sigma algebra 'F_1':
        atom ID
sample         
0             0
1             0
2             1
3             1 

Sigma algebra 'F_2':
        atom ID
sample         
0             0
1             1
2             2
3             3 



#### Accessing the list of $\sigma$-algebras

Get the list of all $\sigma$-algebras:

In [15]:
sigma_algebras = F.sigma_algebras
print(f"Number of sigma-algebras: {len(sigma_algebras)}\n")

for i, alg in enumerate(sigma_algebras):
    print(f"Sigma-algebra {i}: {alg.name}, {alg.num_atoms} atoms")

Number of sigma-algebras: 3

Sigma-algebra 0: F_0, 1 atoms
Sigma-algebra 1: F_1, 2 atoms
Sigma-algebra 2: F_2, 4 atoms
